# Credit Card / Financial Transaction Fraud Detection

## Objective
The objective of this project is to build a machine learning system that can identify fraudulent financial transactions. The PaySim dataset is explored, preprocessed, and used to train multiple classification models. The models are compared using accuracy, precision, recall, and F1-score.


## 1. Import Required Libraries


In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)


## 2. Download and Load the Dataset


In [ ]:
path = kagglehub.dataset_download("ealaxi/paysim1")
print("Dataset downloaded to:")
print(path)


In [ ]:
csv_file = os.path.join(path, "PS_20174392719_1491204439457_log.csv")
df = pd.read_csv(csv_file)
df.head()


## 3. Understand the Dataset


In [ ]:
print("Dataset Shape:", df.shape)
df.info()


In [ ]:
df.describe()


In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


## 4. Exploratory Data Analysis


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='isFraud', data=df)
plt.title('Fraud vs Non-Fraud Transactions')
plt.xlabel('Fraud Status')
plt.ylabel('Number of Transactions')
plt.show()


In [ ]:
fraud_counts = df["isFraud"].value_counts()
fraud_percent = df["isFraud"].value_counts(normalize=True) * 100
print("Transaction Counts:")
print(fraud_counts)

print("\nTransaction Percentage:")
print(fraud_percent)


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='type', order=df['type'].value_counts().index)
plt.title('Distribution of Transaction Types')
plt.xlabel('Transaction Type')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()


## 5. Data Preprocessing

The `type` column is categorical, so it is converted into numerical values using `LabelEncoder`. The target variable is `isFraud`.


In [ ]:
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

print("Encoded transaction types:")
print(dict(zip(le.classes_, le.transform(le.classes_))))


In [ ]:
df.info()


## 6. Correlation Analysis


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm', annot=False)
plt.title('Correlation Matrix')
plt.show()


## 7. Feature and Target Selection

The dataset is sampled to 200,000 records to make model training faster while retaining a representative set of transactions. Stratified splitting is used so that the fraud/non-fraud distribution is preserved in the training and testing sets.


In [ ]:
df_model = df.sample(n=200000, random_state=42)

X = df_model.drop('isFraud', axis=1)
y = df_model['isFraud']

print("Features:", X.shape)
print("Target:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)


## 8. Model 1 - Logistic Regression


In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, zero_division=0))


## 9. Model 2 - Decision Tree


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree")
print("Accuracy :", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, zero_division=0))


## 10. Model 3 - Random Forest


In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest")
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, zero_division=0))


## 11. Model Comparison


In [ ]:
predictions = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf
}

results = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1-Score': f1_score(y_test, pred, zero_division=0)
    }
    for name, pred in predictions.items()
])

results.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)


## 12. Confusion Matrix - Random Forest

For fraud detection, recall is especially important because missing a fraudulent transaction can be costly.


In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## 13. Feature Importance


In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values()

plt.figure(figsize=(9,6))
importance.plot(kind='barh')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()


## 14. Sample Fraud Prediction


In [ ]:
sample = X_test.iloc[[0]]
prediction = rf.predict(sample)[0]
probability = rf.predict_proba(sample)[0]

print("Prediction:", "Fraud" if prediction == 1 else "Genuine")
print("Probability of Genuine:", probability[0])
print("Probability of Fraud  :", probability[1])


# Conclusion

This project developed a machine learning based fraud detection system using the PaySim financial transaction dataset. The data was explored for its structure, missing values, duplicates, transaction types, and fraud distribution. Categorical transaction data was encoded and the dataset was prepared for classification.

Three classification algorithms—Logistic Regression, Decision Tree, and Random Forest—were trained and evaluated. Accuracy, precision, recall, and F1-score were used for comparison. A confusion matrix and Random Forest feature-importance plot were also used to understand model performance and the contribution of different features.

Overall, the project demonstrates how data mining and machine learning techniques can be applied to detect potentially fraudulent financial transactions and support automated fraud monitoring.
